# Protein / carbohydrate audit: steps 1-4

**Goal:** one row per ingredient with protein and carbs **per 100 g**, plus a confidence tier, instead of tuning thresholds.

| Step | What it does |
|---|---|
| 0 | Load the raw table, parse quantities and units (nothing is deleted) |
| 1 | Flag the *fallback profile* (a default nutrient profile applied to thousands of unrelated names) |
| 2 | Convert units to grams and run a mass-balance check |
| 3 | Build the per-100 g ingredient table, with n, spread and a tier |
| 4 | Validate against USDA FoodData Central and report the agreement rate |

Every row keeps its flags, so the raw data stays untouched and every exclusion has a recorded reason.

## Step 0. Load and parse

In [1]:
import re, difflib
import numpy as np
import pandas as pd
from fractions import Fraction
from pathlib import Path

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200)

# ---- EDIT THESE PATHS ----------------------------------------------------
RECIPE_FILE = Path("../dataset/raw/RecipeDB_full/compressed_data.csv.gz")
USDA_SR     = Path("../dataset/external/usda/FoodData_Central_sr_legacy_food_csv_2018-04")
USDA_FF     = Path("../dataset/external/usda/FoodData_Central_foundation_food_csv_2026-04-30")
OUT         = Path("results/nutrient_audit")
# --------------------------------------------------------------------------
OUT.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RECIPE_FILE, low_memory=False)
print("raw shape:", raw.shape)

NUT = ["energy_kcal", "carbohydrates", "protein_g", "fat_g"]
df = raw.copy()
for c in NUT:
    df[c] = pd.to_numeric(df[c], errors="coerce")      # '-' becomes NaN
df["ingredient_name"] = df["ingredient_name"].astype("string").str.strip().str.lower()

def parse_qty(s):
    """'1/2' -> 0.5, '1 1/2' -> 1.5. Ranges/garbage such as '1 -4' -> NaN (ambiguous, do not guess)."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if "-" in s:
        return np.nan
    try:
        return sum(float(Fraction(p)) for p in s.split())
    except (ValueError, ZeroDivisionError):
        return np.nan

df["qty"]  = df["quantity"].map(parse_qty)
df["unit_clean"] = df["unit"].astype("string").str.strip().str.lower()
df["has_nutrients"] = df[NUT].notna().all(axis=1)

print(f"rows with nutrient values: {df.has_nutrients.sum():,} ({df.has_nutrients.mean():.1%})")
print(f"quantity unparseable:      {df.qty.isna().mean():.1%}")

raw shape: (1191643, 10)
rows with nutrient values: 791,500 (66.4%)
quantity unparseable:      9.1%


## Step 1. Detect the fallback profile

Rows are `quantity x per-gram coefficient`, so raw values scale with quantity and a *value* threshold cannot tell good from bad.
Divide each row by its own kcal to get a **scale-free signature** `(protein/kcal, carbs/kcal, fat/kcal)`. A default profile
shows up as one signature shared by an absurd number of unrelated names. Real foods share signatures only with a handful of synonyms.

In [2]:
nz = df[df.has_nutrients & (df.energy_kcal > 0)].copy()
for c in ["protein_g", "carbohydrates", "fat_g"]:
    nz[c + "_per_kcal"] = (nz[c] / nz.energy_kcal).round(3)
nz["signature"] = list(zip(nz.protein_g_per_kcal, nz.carbohydrates_per_kcal, nz.fat_g_per_kcal))

sig = (nz.groupby("signature")
         .agg(distinct_names=("ingredient_name", "nunique"), rows=("signature", "size"))
         .sort_values("distinct_names", ascending=False))
print(sig.head(8))

                      distinct_names   rows
signature                                  
(0.222, 0.0, 0.006)             7368  63527
(0.0, 0.0, 0.113)                 52  35346
(0.0, 0.0, 0.111)                 28   1140
(0.0, 0.25, 0.0)                  21  19332
(0.047, 0.22, 0.011)              12   3169
(0.0, 0.258, 0.0)                 11  13892
(0.202, 0.0, 0.015)               10    281
(0.001, 0.248, 0.0)                9    228


Expect one signature with thousands of distinct names and then a cliff to a few dozen. That gap is the evidence, and it is why no
threshold needs tuning. The next cell sets the fallback cutoff from the data (any signature on >= 500 names) and prints the cliff.

In [3]:
FALLBACK_MIN_NAMES = 500
fb_sigs = set(sig.index[sig.distinct_names >= FALLBACK_MIN_NAMES])
print("fallback signatures:", fb_sigs)
print("largest non-fallback signature has", int(sig.distinct_names[~sig.index.isin(fb_sigs)].max()), "distinct names")

nz["flag_fallback"] = nz.signature.isin(fb_sigs)
df["flag_fallback"] = False
df.loc[nz.index, "flag_fallback"] = nz.flag_fallback

fb = nz[nz.flag_fallback]
print(f"\nfallback rows: {len(fb):,} ({len(fb)/len(nz):.1%} of nutrient rows) on {fb.ingredient_name.nunique():,} names")
print("most common names carrying it:", fb.ingredient_name.value_counts().head(8).to_dict())

fallback signatures: {(0.222, 0.0, 0.006)}
largest non-fallback signature has 52 distinct names

fallback rows: 63,527 (8.5% of nutrient rows) on 7,368 names
most common names carrying it: {'garlic clove': 3286, 'salt pepper': 2251, 'extra virgin olive oil': 1456, 'purpose flour': 1164, 'red pepper flake': 1032, 'salt black pepper': 871, 'red bell pepper': 776, 'kosher salt': 694}


**Two caveats, both checked on your data.**

1. *Do not* flag by "protein is a multiple of 5.61 and carbs = 0" alone. Real meats and fish also have 0 carbs and ~20 g protein per 100 g
   (beef steak, chuck roast, halibut), so that rule deletes valid rows. The signature rule keys on the whole profile.
2. The energy check (4P + 4C + 9F vs kcal) does **not** catch fallback rows, because the fallback profile is internally consistent
   (below). Use it as a sanity check, not as the fallback detector.

In [4]:
nz["energy_est"] = 4*nz.protein_g + 4*nz.carbohydrates + 9*nz.fat_g
nz["energy_rel_err"] = (nz.energy_kcal - nz.energy_est).abs() / nz.energy_kcal
print("median energy error, fallback rows:    ", round(nz[nz.flag_fallback].energy_rel_err.median(), 3))
print("median energy error, non-fallback rows:", round(nz[~nz.flag_fallback].energy_rel_err.median(), 3))
print("share of non-fallback rows with >25% energy error:", round((nz[~nz.flag_fallback].energy_rel_err > .25).mean(), 3))

median energy error, fallback rows:     0.06
median energy error, non-fallback rows: 0.067
share of non-fallback rows with >25% energy error: 0.166


## Step 2. Units to grams, then mass balance

Only mass units give an exact gram weight. Volume and count units (cup, tsp, clove, 'can') depend on the ingredient's density, which the table
does not store, so they are excluded from the per-100 g estimate. The mass-balance rule is physical: protein + carbs + fat cannot exceed the row's mass.

In [5]:
TO_GRAMS = {"g":1, "gram":1, "grams":1, "kg":1000, "kilogram":1000, "kilograms":1000,
            "oz":28.3495, "ounce":28.3495, "ounces":28.3495,
            "lb":453.592, "lbs":453.592, "pound":453.592, "pounds":453.592}

df["mass_g"] = df.unit_clean.map(TO_GRAMS) * df.qty
df.loc[df.mass_g <= 0, "mass_g"] = np.nan
df["has_mass"] = df.mass_g.notna()

m = df.has_nutrients & df.has_mass
df["macro_fraction"] = (df.protein_g + df.carbohydrates + df.fat_g) / df.mass_g
df["flag_mass_balance"] = m & (df.macro_fraction > 1.05)     # 5% slack for rounding

print(f"nutrient rows with a known gram mass: {m.sum():,} ({m.sum()/df.has_nutrients.sum():.1%} of nutrient rows)")
print(f"mass-balance violations: {df.flag_mass_balance.sum():,}")
print(f"  of which already fallback-flagged: {(df.flag_mass_balance & df.flag_fallback).sum():,}")
print("\nmacro_fraction percentiles (mass rows):")
print(df.loc[m, "macro_fraction"].quantile([.5, .9, .99, .999]).round(3).to_string())

nutrient rows with a known gram mass: 68,928 (8.7% of nutrient rows)
mass-balance violations: 452
  of which already fallback-flagged: 257

macro_fraction percentiles (mass rows):
0.500     0.250
0.900     0.856
0.990     1.000
0.999    13.800


## Step 3. Per-100 g table, one row per ingredient

Use only rows that are (a) not fallback, (b) pass mass balance, (c) have a known gram mass. Take the **median** per ingredient and keep
`n` and the IQR so a wide spread reads as "uncertain".

In [6]:
df["usable"] = df.has_nutrients & df.has_mass & ~df.flag_fallback & ~df.flag_mass_balance
u = df[df.usable].copy()
for c in NUT:
    u[c + "_100g"] = u[c] / u.mass_g * 100

def iqr(s): return s.quantile(.75) - s.quantile(.25)

per100 = (u.groupby("ingredient_name")
            .agg(n_rows=("mass_g", "size"),
                 protein_100g=("protein_g_100g", "median"), protein_iqr=("protein_g_100g", iqr),
                 carbs_100g=("carbohydrates_100g", "median"), carbs_iqr=("carbohydrates_100g", iqr),
                 fat_100g=("fat_g_100g", "median"),
                 kcal_100g=("energy_kcal_100g", "median")))

# ingredient frequency over ALL recipe rows (for weighting)
freq = df.ingredient_name.value_counts().rename("recipe_rows")
per100 = per100.join(freq)

def consistent(r, min_n=5):
    ok_p = r.protein_iqr <= max(2.0, 0.20*abs(r.protein_100g))
    ok_c = r.carbs_iqr   <= max(2.0, 0.20*abs(r.carbs_100g))
    return (r.n_rows >= min_n) and ok_p and ok_c
per100["internally_consistent"] = per100.apply(consistent, axis=1)

print("ingredients with a per-100g estimate:", len(per100), "of", df.ingredient_name.nunique())
print("internally consistent:", int(per100.internally_consistent.sum()))
print()
print(per100.sort_values("recipe_rows", ascending=False).head(25).round(1))

ingredients with a per-100g estimate: 1617 of 20282
internally consistent: 685

                 n_rows  protein_100g  protein_iqr  carbs_100g  carbs_iqr  fat_100g  kcal_100g  recipe_rows  internally_consistent
ingredient_name                                                                                                                   
salt                 33           5.1          0.0         0.0        0.0      80.5      748.0        48191                   True
onion               196           1.1          0.0         9.3        0.0       0.1       40.0        35449                   True
butter             1752          17.8          0.0        26.2        0.0      48.0      570.0        30036                   True
water               264           0.0          0.0         0.0        0.0       0.0        0.0        26786                   True
olive oil            67           0.0          0.0         0.0        0.0     100.0      884.0        23797                   True
sug

**Read the table critically.** Internal consistency only tells you the lookup was stable, not that it was correct. For example, a stable-but-wrong
coefficient (a mis-mapped food) has a tiny IQR. Step 4 is what catches those.

## Step 4. Validate against USDA FoodData Central

Build a USDA per-100 g reference from SR Legacy (+ Foundation foods), map the most frequent ingredients to a USDA entry, and measure agreement.
**The mapping is manual on purpose**: it is the auditable human decision. The dict below is a verified starter set; extend it to your top 150-200.

In [7]:
def load_usda(folder, only_type=None):
    food = pd.read_csv(folder / "food.csv", usecols=["fdc_id", "data_type", "description"])
    if only_type:
        food = food[food.data_type == only_type]
    fn = pd.read_csv(folder / "food_nutrient.csv", usecols=["fdc_id", "nutrient_id", "amount"])
    fn = fn[fn.nutrient_id.isin([1003, 1004, 1005, 1008])]
    wide = fn.pivot_table(index="fdc_id", columns="nutrient_id", values="amount", aggfunc="mean")
    wide = wide.rename(columns={1003: "usda_protein", 1004: "usda_fat", 1005: "usda_carbs", 1008: "usda_kcal"})
    return food.merge(wide, left_on="fdc_id", right_index=True)

usda = pd.concat([load_usda(USDA_SR),
                  load_usda(USDA_FF, only_type="foundation_food")], ignore_index=True)
print("USDA reference foods:", len(usda))

USDA reference foods: 8220


In [8]:
USDA_MAP = {
 "garlic clove": "Garlic, raw",
 "garlic": "Garlic, raw",
 "onion": "Onions, raw",
 "red onion": "Onions, raw",
 "white onion": "Onions, raw",
 "yellow onion": "Onions, raw",
 "green onion": "Onions, spring or scallions (includes tops and bulb), raw",
 "scallion": "Onions, spring or scallions (includes tops and bulb), raw",
 "spring onion": "Onions, spring or scallions (includes tops and bulb), raw",
 "shallot": "Shallots, raw",
 "leek": "Leeks, (bulb and lower leaf-portion), raw",
 "chive": "Chives, raw",
 "salt": "Salt, table",
 "kosher salt": "Salt, table",
 "sea salt": "Salt, table",
 "black pepper": "Spices, pepper, black",
 "white pepper": "Spices, pepper, white",
 "cayenne pepper": "Spices, pepper, red or cayenne",
 "cayenne": "Spices, pepper, red or cayenne",
 "butter": "Butter, salted",
 "olive oil": "Oil, olive, salad or cooking",
 "extra virgin olive oil": "Oil, olive, salad or cooking",
 "vegetable oil": "Oil, vegetable, soybean, refined",
 "canola oil": "Oil, canola",
 "sesame oil": "Oil, sesame, salad or cooking",
 "peanut oil": "Oil, peanut, salad or cooking",
 "sugar": "Sugars, granulated",
 "white sugar": "Sugars, granulated",
 "caster sugar": "Sugars, granulated",
 "brown sugar": "Sugars, brown",
 "icing sugar": "Sugars, powdered",
 "confectioner ' sugar": "Sugars, powdered",
 "egg": "Egg, whole, raw, fresh",
 "egg yolk": "Egg, yolk, raw, fresh",
 "egg white": "Egg, white, raw, fresh",
 "tomato": "Tomatoes, red, ripe, raw, year round average",
 "plum tomato": "Tomatoes, red, ripe, raw, year round average",
 "rom tomato": "Tomatoes, red, ripe, raw, year round average",
 "cherry tomato": "Tomatoes, red, ripe, raw, year round average",
 "tomato paste": "Tomato products, canned, paste, without salt added",
 "tomato sauce": "Tomato products, canned, sauce",
 "tomato puree": "Tomato products, canned, puree, without salt added",
 "tomato juice": "Tomato juice, canned, without salt added",
 "milk": "Milk, whole, 3.25% milkfat, with added vitamin D",
 "heavy cream": "Cream, fluid, heavy whipping",
 "heavy whipping cream": "Cream, fluid, heavy whipping",
 "cream": "Cream, fluid, heavy whipping",
 "whipping cream": "Cream, fluid, heavy whipping",
 "buttermilk": "Milk, buttermilk, fluid, cultured, lowfat",
 "flour": "Wheat flour, white, all-purpose, enriched, bleached",
 "purpose flour": "Wheat flour, white, all-purpose, enriched, bleached",
 "plain flour": "Wheat flour, white, all-purpose, enriched, bleached",
 "wheat flour": "Wheat flour, whole-grain",
 "lemon juice": "Lemon juice, raw",
 "lime juice": "Lime juice, raw",
 "lemon": "Lemons, raw, without peel",
 "lime": "Limes, raw",
 "orange": "Oranges, raw, all commercial varieties",
 "orange juice": "Orange juice, raw",
 "lemon zest": "Lemon peel, raw",
 "orange zest": "Orange peel, raw",
 "carrot": "Carrots, raw",
 "potato": "Potatoes, flesh and skin, raw",
 "red potato": "Potatoes, red, flesh and skin, raw",
 "celery": "Celery, raw",
 "celery rib": "Celery, raw",
 "ginger": "Ginger root, raw",
 "gingerroot": "Ginger root, raw",
 "cilantro": "Coriander (cilantro) leaves, raw",
 "cilantro leaf": "Coriander (cilantro) leaves, raw",
 "coriander": "Coriander (cilantro) leaves, raw",
 "coriander leaf": "Coriander (cilantro) leaves, raw",
 "parsley": "Parsley, fresh",
 "basil": "Basil, fresh",
 "basil leaf": "Basil, fresh",
 "thyme": "Thyme, fresh",
 "rosemary": "Rosemary, fresh",
 "mint": "Spearmint, fresh",
 "mint leaf": "Spearmint, fresh",
 "oregano": "Spices, oregano, dried",
 "paprika": "Spices, paprika",
 "chili powder": "Spices, chili powder",
 "cumin": "Spices, cumin seed",
 "cumin seed": "Spices, cumin seed",
 "cinnamon": "Spices, cinnamon, ground",
 "nutmeg": "Spices, nutmeg, ground",
 "clove": "Spices, cloves, ground",
 "turmeric": "Spices, turmeric, ground",
 "turmeric powder": "Spices, turmeric, ground",
 "bay leaf": "Spices, bay leaf",
 "garlic powder": "Spices, garlic powder",
 "onion powder": "Spices, onion powder",
 "curry powder": "Spices, curry powder",
 "allspice": "Spices, allspice, ground",
 "cardamom": "Spices, cardamom",
 "fennel seed": "Spices, fennel seed",
 "coriander seed": "Spices, coriander seed",
 "caraway seed": "Spices, caraway seed",
 "dill": "Dill weed, fresh",
 "red pepper flake": "Spices, pepper, red or cayenne",
 "marjoram": "Spices, marjoram, dried",
 "soy sauce": "Soy sauce made from soy and wheat (shoyu)",
 "fish sauce": "Sauce, fish, ready-to-serve",
 "worcestershire sauce": "Sauce, worcestershire",
 "hoisin sauce": "Sauce, hoisin, ready-to-serve",
 "oyster sauce": "Sauce, oyster, ready-to-serve",
 "salsa": "Sauce, salsa, ready-to-serve",
 "ketchup": "Catsup",
 "mayonnaise": "Salad dressing, mayonnaise, regular",
 "chili sauce": "Sauce, tomato chili sauce, bottled, with salt",
 "baking powder": "Leavening agents, baking powder, double-acting, sodium aluminum sulfate",
 "baking soda": "Leavening agents, baking soda",
 "cornstarch": "Cornstarch",
 "vanilla extract": "Vanilla extract",
 "vanilla": "Vanilla extract",
 "honey": "Honey",
 "maple syrup": "Syrups, maple",
 "molasses": "Molasses",
 "vinegar": "Vinegar, cider",
 "white vinegar": "Vinegar, distilled",
 "cider vinegar": "Vinegar, cider",
 "balsamic vinegar": "Vinegar, balsamic",
 "red wine vinegar": "Vinegar, red wine",
 "white wine": "Alcoholic beverage, wine, table, white",
 "red wine": "Alcoholic beverage, wine, table, red",
 "beef": "Beef, ground, 80% lean meat / 20% fat, raw",
 "chicken breast": "Chicken, broiler or fryers, breast, skinless, boneless, meat only, raw",
 "chicken breast half": "Chicken, broiler or fryers, breast, skinless, boneless, meat only, raw",
 "chicken": "Chicken, broilers or fryers, meat and skin, raw",
 "chicken thigh": "Chicken, broilers or fryers, thigh, meat and skin, raw",
 "pork": "Pork, fresh, loin, whole, separable lean and fat, raw",
 "ham": "Ham, sliced, regular",
 "bacon": "Pork, cured, bacon, unprepared",
 "lamb": "Lamb, ground, raw",
 "turkey": "Turkey, whole, meat and skin, raw",
 "shrimp": "Crustaceans, shrimp, raw",
 "parmesan cheese": "Cheese, parmesan, grated",
 "mozzarella cheese": "Cheese, mozzarella, whole milk",
 "cheddar cheese": "Cheese, cheddar",
 "sharp cheddar cheese": "Cheese, cheddar",
 "cream cheese": "Cheese, cream",
 "feta cheese": "Cheese, feta",
 "swiss cheese": "Cheese, swiss",
 "cottage cheese": "Cheese, cottage, creamed, large or small curd",
 "ricotta cheese": "Cheese, ricotta, whole milk",
 "monterey jack cheese": "Cheese, monterey",
 "cheese": "Cheese, cheddar",
 "yogurt": "Yogurt, plain, whole milk",
 "plain yogurt": "Yogurt, plain, whole milk",
 "coconut milk": "Nuts, coconut milk, raw",
 "coconut": "Nuts, coconut meat, raw",
 "margarine": "Margarine, regular, hard, soybean (hydrogenated)",
 "almond": "Nuts, almonds",
 "walnut": "Nuts, walnuts, english",
 "pecan": "Nuts, pecans",
 "peanut": "Peanuts, all types, raw",
 "pine nut": "Nuts, pine nuts, dried",
 "peanut butter": "Peanut butter, smooth style, without salt",
 "sesame seed": "Seeds, sesame seeds, whole, dried",
 "cocoa powder": "Cocoa, dry powder, unsweetened",
 "rice": "Rice, white, long-grain, regular, raw, enriched",
 "white rice": "Rice, white, long-grain, regular, raw, enriched",
 "brown rice": "Rice, brown, long-grain, raw",
 "basmati rice": "Rice, white, long-grain, regular, raw, enriched",
 "pasta": "Pasta, dry, enriched",
 "spaghetti": "Pasta, dry, enriched",
 "bread": "Bread, white wheat",
 "french bread": "Bread, french or vienna",
 "cornmeal": "Cornmeal, whole-grain, yellow",
 "oat": "Oats",
 "raisin": "Raisins, dark, seedless",
 "black bean": "Beans, black, mature seeds, raw",
 "kidney bean": "Beans, kidney, all types, mature seeds, raw",
 "garbanzo bean": "Chickpeas (garbanzo beans, bengal gram), mature seeds, raw",
 "chickpea": "Chickpeas (garbanzo beans, bengal gram), mature seeds, raw",
 "lentil": "Lentils, raw",
 "tofu": "Tofu, raw, firm, prepared with calcium sulfate",
 "green bean": "Beans, snap, green, raw",
 "pea": "Peas, green, raw",
 "corn": "Corn, sweet, yellow, raw",
 "bean sprout": "Mung beans, mature seeds, sprouted, raw",
 "mushroom": "Mushrooms, white, raw",
 "spinach": "Spinach, raw",
 "broccoli": "Broccoli, raw",
 "cabbage": "Cabbage, raw",
 "lettuce": "Lettuce, cos or romaine, raw",
 "cucumber": "Cucumber, with peel, raw",
 "zucchini": "Squash, summer, zucchini, includes skin, raw",
 "eggplant": "Eggplant, raw",
 "avocado": "Avocados, raw, all commercial varieties",
 "red bell pepper": "Peppers, sweet, red, raw",
 "green bell pepper": "Peppers, sweet, green, raw",
 "green pepper": "Peppers, sweet, green, raw",
 "bell pepper": "Peppers, sweet, green, raw",
 "jalapeno": "Peppers, jalapeno, raw",
 "jalapeno pepper": "Peppers, jalapeno, raw",
 "red pepper": "Peppers, sweet, red, raw",
 "olive": "Olives, ripe, canned (small-extra large)",
 "black olive": "Olives, ripe, canned (small-extra large)",
 "caper": "Capers, canned",
 "banana": "Bananas, raw",
 "apple": "Apples, raw, with skin",
 "pineapple": "Pineapple, raw, all varieties",
 "mango": "Mangos, raw",
 "strawberry": "Strawberries, raw",
 "apricot": "Apricots, raw",
 "chicken broth": "Soup, chicken broth, canned, condensed",
 "chicken stock": "Soup, stock, chicken, home-prepared",
 "beef broth": "Soup, beef broth or bouillon canned, ready-to-serve",
 "beef stock": "Soup, stock, beef, home-prepared",
 "vegetable broth": "Soup, vegetable broth, ready to serve",
 "vegetable stock": "Soup, vegetable broth, ready to serve",
 "water": "Beverages, water, tap, drinking",
  "flat leaf parsley": "Parsley, fresh",
 "italian parsley": "Parsley, fresh",
 "green chilies": "Peppers, hot chili, green, raw",
 "red chilies": "Peppers, hot chili, red, raw",
 "breadcrumb": "Bread, white, commercially prepared (includes soft bread crumbs)",
 "flour tortilla": "Tortillas, ready-to-bake or -fry, flour, shelf stable",
 "corn tortilla": "Tortillas, ready-to-bake or -fry, corn",
 "mustard": "Mustard, prepared, yellow",
 "dijon mustard": "Mustard, prepared, yellow",
 "yeast": "Leavening agents, yeast, baker's, active dry",
 "active yeast": "Leavening agents, yeast, baker's, active dry",
 "black peppercorn": "Spices, pepper, black",
 "italian sausage": "Sausage, Italian, pork, mild, raw",
 "shortening": "Shortening, household, soybean (partially hydrogenated)-cottonseed (partially hydrogenated)",
 "sage": "Spices, sage, ground",
 "mustard seed": "Spices, mustard seed, ground",
 "kalamata olive": "Olives, ripe, canned (small-extra large)",
 "green olive": "Olives, pickled, canned or bottled, green",
 "cream mushroom soup": "Soup, cream of mushroom, canned, condensed",
 "beer": "Alcoholic beverage, beer, regular, all",
 "rum": "Alcoholic beverage, distilled, all (gin, rum, vodka, whiskey) 86 proof",
 "brandy": "Alcoholic beverage, distilled, all (gin, rum, vodka, whiskey) 86 proof",
 "lemongrass": "Lemon grass (citronella), raw",
 "taco seasoning": "Seasoning mix, dry, taco, original",
 "tortilla chip": "Snacks, tortilla chips, plain, white corn, salted",
 "ghee": "Butter oil, anhydrous",
 "coffee": "Beverages, coffee, brewed, prepared with tap water",
 "cornflour": "Cornstarch",
 "self raising flour": "Wheat flour, white, all-purpose, self-rising, enriched",
 "pineapple juice": "Pineapple juice, canned or bottled, unsweetened, with added ascorbic acid",
 "pinto bean": "Beans, pinto, mature seeds, raw",
 "button mushroom": "Mushrooms, white, raw",
 "spaghetti sauce": "Sauce, pasta, spaghetti/marinara, ready-to-serve",
 "corn kernel": "Corn, sweet, yellow, raw",
 "kernel corn": "Corn, sweet, yellow, raw",
 "grain rice": "Rice, white, long-grain, regular, raw, enriched",
 "cold water": "Beverages, water, tap, drinking",
 "hot water": "Beverages, water, tap, drinking",
}
def resolve(desc):
    exact = usda[usda.description == desc]
    if len(exact): return exact.iloc[0]
    pre = usda[usda.description.str.startswith(desc)]
    return pre.iloc[0] if len(pre) else None

rows, missing = [], []
for name, desc in USDA_MAP.items():
    r = resolve(desc)
    if r is None:
        missing.append((name, desc)); continue
    rows.append({"ingredient_name": name, "usda_description": r.description,
                 "usda_protein": r.usda_protein, "usda_carbs": r.usda_carbs,
                 "usda_fat": r.usda_fat, "usda_kcal": r.usda_kcal})
ref = pd.DataFrame(rows).set_index("ingredient_name")
print("mapped:", len(ref), "| unresolved USDA descriptions (fix these):", missing)

mapped: 251 | unresolved USDA descriptions (fix these): []


In [9]:
TOL_ABS, TOL_REL = 2.0, 0.20       # agree if within max(2 g, 20% of USDA value)

chk = per100.join(ref, how="inner")
def agree(est, ref_):
    return (est - ref_).abs() <= np.maximum(TOL_ABS, TOL_REL * ref_.abs())
chk["protein_ok"] = agree(chk.protein_100g, chk.usda_protein)
chk["carbs_ok"]   = agree(chk.carbs_100g,   chk.usda_carbs)
chk["both_ok"]    = chk.protein_ok & chk.carbs_ok

print(f"checked: {len(chk)} ingredients")
print(f"protein agrees: {chk.protein_ok.mean():.0%} | carbs agrees: {chk.carbs_ok.mean():.0%} | both: {chk.both_ok.mean():.0%}")
cols = ["n_rows","protein_100g","usda_protein","carbs_100g","usda_carbs","protein_ok","carbs_ok","recipe_rows"]
print("\nDISAGREEMENTS (these are the mapping / lookup errors):")
print(chk[~chk.both_ok].sort_values("recipe_rows", ascending=False)[cols].round(1))
chk.to_csv(OUT / "usda_agreement.csv")

checked: 209 ingredients
protein agrees: 85% | carbs agrees: 82% | both: 78%

DISAGREEMENTS (these are the mapping / lookup errors):
                   n_rows  protein_100g  usda_protein  carbs_100g  usda_carbs  protein_ok  carbs_ok  recipe_rows
ingredient_name                                                                                                 
salt                   33           5.1           0.0         0.0         0.0       False      True        48191
butter               1752          17.8           0.8        26.2         0.1       False     False        30036
flour                 143           6.0          10.3        80.1        76.3       False      True        10400
cinnamon                8           4.5           4.0        48.6        80.6        True     False        10118
parmesan cheese       349          40.0          28.4        40.0        13.9       False     False         8219
mushroom             1092           2.2           3.1         6.8         3.

Disagreement is the useful output: those are ingredients where the RecipeDB lookup pointed at the wrong food (a stable but wrong coefficient). Each one
is a documented error rate for your report, and the correct USDA value can replace it.

Next cell: suggest USDA candidates for your top unmapped names so extending `USDA_MAP` to 150-200 is a review job, not a search job.

In [10]:
top_unmapped = [n for n in freq.index[:400] if n not in USDA_MAP and isinstance(n, str)][:200]
choices = usda.description.tolist()
sugg = []
for n in top_unmapped:
    cand = difflib.get_close_matches(n.title(), choices, n=3, cutoff=0.3)
    # prefer raw / plain entries by sorting shorter descriptions first
    cand = sorted(cand, key=len)
    sugg.append({"ingredient_name": n, "recipe_rows": int(freq[n]),
                 "suggestion_1": cand[0] if len(cand) > 0 else "",
                 "suggestion_2": cand[1] if len(cand) > 1 else "",
                 "suggestion_3": cand[2] if len(cand) > 2 else ""})
pd.DataFrame(sugg).to_csv(OUT / "usda_mapping_suggestions.csv", index=False)
print("wrote", OUT / "usda_mapping_suggestions.csv", "- review it, copy the correct ones into USDA_MAP, re-run Step 4")

wrote results/nutrient_audit/usda_mapping_suggestions.csv - review it, copy the correct ones into USDA_MAP, re-run Step 4


## Step 4a. Which USDA food did RecipeDB actually use?

RecipeDB's per-100 g profiles look like USDA SR values. If so, each ingredient's profile should match one USDA entry *exactly*. Reverse-matching shows
**what the source's lookup really picked**, which turns "this value looks off" into "`salt` was matched to `salt pork`", a checkable claim.
Caveat: pure-fat profiles (100 g fat, 884 kcal) tie across many oils, so ties are flagged rather than named.

In [11]:
usda_ok = usda.dropna(subset=["usda_protein", "usda_carbs", "usda_fat"]).copy()
U = usda_ok[["usda_protein", "usda_carbs", "usda_fat"]].to_numpy()

def reverse_match(r, tol=0.5):
    d = np.sqrt(((U - np.array([r.protein_100g, r.carbs_100g, r.fat_100g]))**2).sum(axis=1))
    hit = np.where(d <= tol)[0]
    if len(hit) == 0:
        return pd.Series({"matched_usda": "(no exact match)", "n_ties": 0})
    best = hit[np.argsort(d[hit])]
    return pd.Series({"matched_usda": usda_ok.iloc[best[0]].description[:60], "n_ties": len(hit)})

top = per100[per100.n_rows >= 3].sort_values("recipe_rows", ascending=False).head(200)
rm = top.join(top.apply(reverse_match, axis=1))
rm["ambiguous_tie"] = rm.n_ties > 3
review = rm[["recipe_rows", "n_rows", "protein_100g", "carbs_100g", "fat_100g", "matched_usda", "n_ties", "ambiguous_tie"]]
review.to_csv(OUT / "reverse_match_review.csv")
print(review.head(40).round(1))

                 recipe_rows  n_rows  protein_100g  carbs_100g  fat_100g                                       matched_usda  n_ties  ambiguous_tie
ingredient_name                                                                                                                                   
salt                   48191      33           5.1         0.0      80.5                        Pork, cured, salt pork, raw       1          False
onion                  35449     196           1.1         9.3       0.1                                        Onions, raw      16           True
butter                 30036    1752          17.8        26.2      48.0  Seeds, sesame butter, tahini, from raw and sto...       1          False
water                  26786     264           0.0         0.0       0.0   Beverages, water, bottled, non-carbonated, EVIAN      74           True
olive oil              23797      67           0.0         0.0     100.0                Oil, whale, bowhead (Alaska Na

Read `reverse_match_review.csv` top to bottom (about 15 minutes for 200 rows). Anything where `matched_usda` is a different food from the ingredient name is a
documented lookup error. Those ingredients should be remapped to the correct USDA entry, not kept.

## Step 4b. Final tiers and weighted coverage

* **trusted**: consistent in the data **and** within tolerance of USDA for both protein and carbs.
* **uncertain**: has an estimate but is unverified (not in the USDA map) or disagrees with USDA, or is inconsistent.
* **unavailable**: no usable row (all rows fallback, mass-balance failures, missing, or no gram-mass unit).

Coverage is weighted by how often each ingredient appears, since that is the number that matters for substitution on real recipes.

In [12]:
final = per100.copy()
final = final.join(chk[["usda_protein", "usda_carbs", "both_ok"]], how="left")
def tier(r):
    if r.both_ok == True:   # NaN (unverified) compares False
        return "trusted" if r.internally_consistent else "uncertain"
    return "uncertain"
final["tier"] = final.apply(tier, axis=1)

all_names = freq.to_frame()
all_names = all_names.join(final[["tier"]], how="left")
all_names["tier"] = all_names.tier.fillna("unavailable")

summary = (all_names.groupby("tier")
           .agg(ingredients=("recipe_rows", "size"), recipe_rows=("recipe_rows", "sum")))
summary["share_of_ingredients"] = summary.ingredients / summary.ingredients.sum()
summary["share_of_recipe_rows"] = summary.recipe_rows / summary.recipe_rows.sum()
print(summary.round(3))

final.to_csv(OUT / "ingredient_per100g_with_tiers.csv")
df.drop(columns=["recipe_title"]).to_csv(OUT / "row_level_flags.csv.gz", index=False, compression="gzip")
print("saved to", OUT)

             ingredients  recipe_rows  share_of_ingredients  share_of_recipe_rows
tier                                                                             
trusted              131       493666                 0.006                 0.415
unavailable        18665       265190                  0.92                 0.223
uncertain           1486       431333                 0.073                 0.362
saved to results/nutrient_audit


In [13]:
rows = []
for name, desc in USDA_MAP.items():
    r = resolve(desc)
    if r is None:
        continue
    rows.append({"ingredient_name": name, "protein_100g": r.usda_protein, "carbs_100g": r.usda_carbs,
                 "fat_100g": r.usda_fat, "kcal_100g": r.usda_kcal, "source": "usda", "tier": "trusted",
                 "usda_description": r.description})
final_usda = pd.DataFrame(rows).set_index("ingredient_name")

fallback = per100[per100.internally_consistent & ~per100.index.isin(final_usda.index)][
    ["protein_100g", "carbs_100g", "fat_100g", "kcal_100g"]].copy()
fallback["source"], fallback["tier"], fallback["usda_description"] = "recipedb_unverified", "uncertain", ""

nutrients_final = pd.concat([final_usda, fallback])
nutrients_final = nutrients_final.join(freq, how="left")
nutrients_final.to_csv(OUT / "ingredient_nutrients_final.csv")

allx = freq.to_frame().join(nutrients_final[["tier"]], how="left")
allx["tier"] = allx.tier.fillna("unavailable")
cov = allx.groupby("tier").agg(ingredients=("recipe_rows", "size"), recipe_rows=("recipe_rows", "sum"))
cov["share_of_recipe_rows"] = (cov.recipe_rows / cov.recipe_rows.sum()).round(3)
print(cov)

             ingredients  recipe_rows  share_of_recipe_rows
tier                                                       
trusted              251       882477                 0.741
unavailable        19517       215477                 0.181
uncertain            514        92235                 0.077
